# 測試：contraction 加分項、過熱標記、區間寬度

In [1]:
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, momentum_scanner as ms
idx=pd.bdate_range(end='2026-10-23',periods=300)
p=ms.Params()

In [2]:
# 1. 過熱：後段拋物線上漲 -> overheated；緩漲 -> 不過熱
base=np.full(300,50.0); base[:200]=50+np.arange(200)*0.05
para=base.copy(); para[240:]=base[239]*np.exp(np.linspace(0,1.0,60))   # 最後 60 天翻倍以上
slow=50+np.arange(300)*0.05
def feats(a): 
    s=pd.Series(a,index=idx); return ms.consolidation_features(s,s*1.01,s*0.99,pd.Series(1e6,index=idx),p)
fp,fs=feats(para),feats(slow)
print(fp['dist_sma50'],fs['dist_sma50'])
assert fp['overheated'] and not fs['overheated']
# 門檻可調
assert not ms.consolidation_features(pd.Series(para,index=idx),pd.Series(para*1.01,index=idx),pd.Series(para*.99,index=idx),pd.Series(1e6,index=idx),
        ms.Params(overheat_dist=5.0))['overheated']

0.47037715307209305 0.01922322479403693


In [3]:
# 2. range_20 = 近 20 日 (最高-最低)/現價
s=pd.Series(100+np.sin(np.arange(300)/3)*5,index=idx); h,l=s*1.01,s*0.99
f=ms.consolidation_features(s,h,l,pd.Series(1e6,index=idx),p)
exp=(h.iloc[-20:].max()-l.iloc[-20:].min())/s.iloc[-1]
print(f['range_20'],exp); assert abs(f['range_20']-exp)<1e-12

0.12409890716330926 0.12409890716330926


In [4]:
# 3. rank_table 排序與門檻：核心分數 -> 不過熱 -> 有收斂 -> 趨勢模板
df=pd.DataFrame({'candidate':True,'core_score':[5,5,5,5,4,3],'overheated':[False,True,False,False,False,False],
 'contraction':[False,False,True,False,True,True],'tt_count':[8,8,8,5,8,8],'avg_pct':[.9]*6},index=list('ABCDEF'))
print(ms.rank_table(df,4).index.tolist())
assert ms.rank_table(df,4).index.tolist()==['C','A','D','B','E']   # F 核心只有 3，被濾掉
assert ms.rank_table(df,3).index.tolist()[-1]=='F'
df.loc['A','candidate']=False
assert 'A' not in ms.rank_table(df,4).index                         # 流動性/強勢仍是硬條件

['C', 'A', 'D', 'B', 'E']


In [5]:
# 4. scan：cons_score = core_score + contraction；passed 沒變（仍要求六條全過）
rng=np.random.default_rng(3); tks=[f'T{i}' for i in range(60)]
c=pd.DataFrame({t:100*np.exp(np.cumsum(rng.normal(0.003,0.015,300))) for t in tks},index=idx)
P=ms.Prices(c,c*1.01,c*.99,pd.DataFrame(5e6,index=idx,columns=tks)*1.0)
r=ms.scan(P,params=ms.Params(min_price=0,min_dollar_vol=0))
assert (r.cons_score-r.core_score==r.contraction.astype(int)).all()
assert (r.passed<=(r.core_score==5)&r.contraction).all()
print('ok')

ok
